# Module 7: Results, Analysis & Ablation Studies

**Estimated time: 45 minutes**

This module dives into the paper's experimental results. We analyze each table and figure, understand what the numbers mean, and explore the ablation studies.

## 7.1 Single-Task Results (Table 1)

Open the paper to **Table 1** (page 6). This shows Objective 1: adapting LLM experts to a single task.

| Task Category | Tasks | Best Baseline | Model Swarms | Avg Improvement |
|--------------|-------|---------------|-------------|-------------|
| Knowledge | MMLU, MMLU-pro, HellaSwag | Varies | SOTA on all 3 | +4.9% |
| Reasoning | GSM8k, K-Crosswords, NLGraph | Varies | SOTA on all 3 | +21.0% |
| Safety | TruthfulQA, RealToxicity, AbstainQA | Varies | SOTA on all 3 | +14.1% |

Key observations:
1. **Reasoning sees the largest gains (+21%).** These tasks require combining multiple capabilities (math + logic + formatting), and the swarm excels at finding synergistic weight configurations.
2. **Knowledge sees the smallest gains (+4.9%).** Closer to memorization/recall — experts already have what they need.
3. **GSM8k improvement is 29.7%** — the single largest improvement. Mathematical reasoning + linguistic understanding reinforce each other.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Paper results: improvement over best baseline by category
categories = ['Knowledge\n(MMLU, MMLU-pro,\nHellaSwag)', 
              'Reasoning\n(GSM8k, K-Cross,\nNLGraph)', 
              'Safety\n(TruthfulQA, RealTox,\nAbstainQA)']
improvements = [4.9, 21.0, 14.1]
colors = ['#42A5F5', '#66BB6A', '#FFA726']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(categories, improvements, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
ax.set_ylabel('Average Improvement over Best Baseline (%)', fontsize=12)
ax.set_title('Model Swarms: Single-Task Improvements by Category', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

for bar, imp in zip(bars, improvements):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'+{imp}%', ha='center', va='bottom', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

## 7.2 Multi-Task Results (Table 2)

The most interesting finding is **Pareto-optimal experts** — the best expert for a domain is not the best for either sub-task individually.

| Domain | Improvement |
|--------|------------|
| Legal | +10.9% |
| Medical | +4.2% |
| Science | +3.8% |
| Culture | +4.0% |

Legal sees the largest gain, possibly because it requires combining language understanding (hearsay detection) with structural knowledge (citation patterns).

In [ ]:
# Multi-task results visualization
domains = ['Legal', 'Medical', 'Science', 'Culture']
improvements = [10.9, 4.2, 3.8, 4.0]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(domains, improvements, color='#7E57C2', height=0.5)
ax.set_xlabel('Improvement over Best Baseline (%)')
ax.set_title('Multi-Task Domain Adaptation Results')
ax.grid(True, alpha=0.3, axis='x')

for bar, imp in zip(bars, improvements):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2.,
            f'+{imp}%', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7.3 Reward Model Results (Table 4)

This demonstrates "steerability" — the algorithm genuinely adapts to whatever reward function you specify.

| Method | General RM | Verbose RM | Concise RM |
|--------|-----------|-----------|------------|
| SLERP | Good | Good | Poor |
| DARE-TIES | Moderate | Good | Poor |
| Model Swarms | **Best** | **Best** | **Best** |

Most baselines inadvertently favor verbosity. Only Model Swarms achieves SOTA on concise AND verbose reward models simultaneously.

**Practical implication**: You can produce specialized models for different user preferences from the same expert pool — just change the utility function.

## 7.4 Human Interest Results (Table 3)

16 niche topics, only 25 validation examples each. **Average win rate: 70.8%** against best baseline.

| Performance Band | Topics | Examples |
|-----------------|--------|----------|
| Strong win (>80%) | 4 | Electric vehicles, PhD applications |
| Moderate win (60-80%) | 7 | Indoor gardening, board games |
| Marginal win (50-60%) | 3 | Cocktail recipes, pet care |
| Loss (<50%) | 2 | Very niche factual topics |

Topics where Model Swarms loses require specific factual knowledge that no expert possesses — confirming that **the search adapts existing knowledge, it doesn't create new knowledge**.

## 7.5 Correctness Emergence (Section 4.5)

This is perhaps the paper's most scientifically interesting finding.

- **C-surge**: % of questions where the final model is correct AND at least one initial expert was wrong → ~48%
- **C-emerge**: % of questions where the final model is correct AND ALL initial experts were wrong → ~44%

**44% C-emerge means**: For almost half the problems that NO expert could solve, the combined model CAN solve them.

In [ ]:
# Visualize correctness emergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# C-surge and C-emerge ranges
metrics = ['C-surge\n(\u22651 expert wrong)', 'C-emerge\n(ALL experts wrong)']
means = [48, 44]
ranges = [(40, 55), (36, 53.5)]

ax1.bar(metrics, means, color=['#26A69A', '#EF5350'], width=0.4, 
        yerr=[[m-r[0] for m, r in zip(means, ranges)],
              [r[1]-m for m, r in zip(means, ranges)]],
        capsize=8)
ax1.set_ylabel('Percentage (%)')
ax1.set_title('Correctness Emergence Metrics')
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_ylim(0, 65)

# Conceptual diagram: how emergence works
labels = ['Expert A\nknows X', 'Expert B\nknows Y', 'Combined\nknows X+Y\n\u2192 solves Z']
sizes = [30, 30, 45]
colors_pie = ['#42A5F5', '#66BB6A', '#FFA726']
ax2.bar(labels, sizes, color=colors_pie, width=0.5)
ax2.set_ylabel('Capability (conceptual)')
ax2.set_title('How Emergence Works: Synergistic Combination')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Possible Mechanisms for Emergence

1. **Cross-capability activation**: Expert A has knowledge X, Expert B has reasoning pattern Y. Combined weights create a pathway where X feeds into Y.
2. **Destructive interference removal**: Some expert weights suppress correct answers. The search finds configurations that reduce these suppressions.
3. **Distributional alignment**: The combined model's token distribution better matches expected output format.

## 7.6 Diamond in the Rough (Section 4.6)

- 89.6% of final best particles were NOT the initially highest-performing expert
- 56.9% started in the bottom half of rankings

**Implication**: Don't prune "weak" experts. Their weakness on the target task doesn't mean they lack valuable capabilities.

## 7.7 Ablation Studies (Table 5)

| Ablation | Performance Impact |
|----------|-------------------|
| Remove randomness | -5 to -15% (LARGEST) |
| Remove repulsion | -5 to -10% |
| Reduce particles (10\u21925) | -5 to -12% |
| Remove particle restart | -3 to -8% |
| Zero initial velocity | -2 to -5% |

**Key takeaways:**
1. **Randomness is essential** — without it, particles collapse to one point
2. **Repulsion helps more than expected** — prevents convergence to bad regions
3. **More particles is better** — but diminishing returns after 10\u219220

In [ ]:
# Ablation study visualization
ablations = [
    'Remove\nrandomness', 'Remove\nrepulsion', 'Reduce\nparticles\n(10\u21925)',
    'Remove\nrestart', 'Zero initial\nvelocity'
]
# Using midpoints of ranges
impacts = [-10, -7.5, -8.5, -5.5, -3.5]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#EF5350' if x < -7 else '#FFA726' if x < -5 else '#66BB6A' for x in impacts]
bars = ax.bar(ablations, impacts, color=colors, width=0.5)
ax.set_ylabel('Performance Impact (%)')
ax.set_title('Ablation Study: Impact of Removing Each Component')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linewidth=0.5)

for bar, impact in zip(bars, impacts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() - 0.5,
            f'{impact}%', ha='center', va='top', fontweight='bold', fontsize=10, color='white')

plt.tight_layout()
plt.show()

## 7.8 Diversity Analysis (Section 4.7)

Total particles fixed at 20. Vary number of distinct experts:

| Configuration | Distinct Experts | Performance |
|---|---|---|
| 1\u00d720 | 1 (duplicated 20\u00d7) | Baseline |
| 2\u00d710 | 2 | +10% |
| 5\u00d74 | 5 | +25% |
| 10\u00d72 | 10 | +33% |
| 10+10 interpolated | 10 | +35% (paper default) |

Performance increases monotonically with diversity. The gain from 1 to 10 distinct experts is ~35%.

In [ ]:
# Diversity analysis
configs = ['1\u00d720', '2\u00d710', '5\u00d74', '10\u00d72', '10+10\ninterp.']
distinct_experts = [1, 2, 5, 10, 10]
relative_performance = [0.0, 10, 25, 33, 35]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(len(configs)), relative_performance, 'o-', linewidth=2.5, 
        markersize=10, color='#7E57C2')
ax.set_xticks(range(len(configs)))
ax.set_xticklabels(configs)
ax.set_xlabel('Particle Configuration (total always 20)')
ax.set_ylabel('Relative Performance Gain (%)')
ax.set_title('Effect of Expert Diversity on Search Performance')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 7.1: Results Interpretation

1. On which task does Model Swarms have the SMALLEST advantage? Why?
2. On which task does it have the LARGEST advantage? What makes that task different?
3. If you had limited compute, which tasks would justify using Model Swarms vs. simpler baselines?

*Your answers:*

1. 
2. 
3. 

## Exercise 7.2: Design Your Own Ablation

Propose an ablation study NOT in the paper. State: (1) hypothesis, (2) experimental setup, (3) success metric.

*Your proposal:*



## Exercise 7.3: Critical Analysis

Write a 3-4 paragraph critical review of the paper's methodology. Consider:
1. Are baselines fair and comprehensive?
2. Are evaluation metrics appropriate?
3. What confounding factors might explain results?
4. What experiments are missing?

*Your review:*



---

**Next: [Module 8 — Advanced Topics: Token Swarms, Extensions & Open Problems](module_08_advanced_topics.ipynb)**